[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tasmia008/Complete_Guidelines_LLM_FineTuning/blob/main/14b_model_merging.ipynb)

# Model merging — TIES, DARE, SLERP, model soups (mergekit)

> **Fine-tuning series — 14b of 26.** A standalone notebook in a set; see `00_README.md` for the full index and order. Conceptual/reference — no GPU setup needed.


## Model merging (TIES · DARE · SLERP · model soups)

Not fine-tuning — **post-training composition**. You combine several *already-trained* checkpoints
directly in **weight space**, with no gradient training and often no data at all. Why bother: fuse
complementary skills (a code fine-tune + a math fine-tune into one model), average several runs for
robustness, or recover generality that specialization eroded. The standard tool is **mergekit**
(`arcee-ai/mergekit`).

Requirement: every input must share the same **architecture and tokenizer** — in practice, fine-tunes
of the *same base model*. A `base_model` is used to compute each model's *delta* (its "task vector").

| Method | Idea | # models | Use when |
|---|---|---|---|
| **Linear / model soup** | weighted average of weights | 2+ | average runs of the *same* recipe for robustness |
| **SLERP** | spherical interpolation along an arc | exactly **2** | smooth blend of two fine-tunes |
| **Task arithmetic** | add/subtract task vectors (finetuned − base) | 2+ | compose or *remove* a skill |
| **TIES** | trim small deltas, elect a sign, then merge | **many** | fuse several task models, less interference |
| **DARE** (`dare_ties` / `dare_linear`) | randomly drop + rescale deltas, then merge | **many** | merge many with even less interference |

In [ ]:
!pip install mergekit

### SLERP — blend two models

Spherical interpolation between exactly two models. `t` is the blend (0 = all A, 1 = all B) and can
be set per layer-type.

```yaml
# slerp.yml
slices:
  - sources:
      - { model: ./model_A, layer_range: [0, 24] }
      - { model: ./model_B, layer_range: [0, 24] }
merge_method: slerp
base_model: ./model_A
parameters:
  t:
    - filter: self_attn
      value: 0.5
    - filter: mlp
      value: 0.3
    - value: 0.5          # default for all other layers
dtype: bfloat16
```

In [ ]:
!mergekit-yaml slerp.yml ./merged-slerp

### TIES / DARE — merge many task models

Fuse several task-specific fine-tunes onto the base. `density` keeps the top fraction of each
model's deltas; `weight` is its contribution (aim for the weights to sum ≈ 1). Switch
`merge_method` to `dare_ties` to add DARE's random-drop-and-rescale step.

```yaml
# ties.yml
models:
  - model: ./base                                 # base: no params (used to compute deltas)
  - model: ./finetune_code
    parameters: { density: 0.5, weight: 0.5 }
  - model: ./finetune_math
    parameters: { density: 0.5, weight: 0.5 }
merge_method: ties                                # or: dare_ties
base_model: ./base
parameters:
  normalize: true
  int8_mask: true
dtype: bfloat16
```

In [ ]:
!mergekit-yaml ties.yml ./merged-ties

### Linear — the "model soup"

The simplest merge: a weighted average. Classic use is averaging several runs of the *same* recipe
(different seeds/checkpoints) for a small, free robustness gain.

```yaml
# soup.yml
models:
  - model: ./run_seed1
    parameters: { weight: 0.5 }
  - model: ./run_seed2
    parameters: { weight: 0.5 }
merge_method: linear
dtype: bfloat16
```

In [ ]:
!mergekit-yaml soup.yml ./merged-soup

### Notes

- Inputs must share the same **architecture & tokenizer** (same base family).
- **Merging LoRA adapters?** Merge each adapter into its base first (notebook 17), then merge the
  resulting full models here.
- Merging is **cheap** (CPU, minutes, no training data) but **empirical** — always evaluate the
  merge against its parent models; nothing guarantees it's better.
- Knobs: `density` = fraction of deltas kept (TIES/DARE) · `weight` = each model's contribution
  (sum ≈ 1) · `t` = SLERP interpolation.
- Beyond merging, mergekit can also assemble a **Mixture-of-Experts** from several models
  (`mergekit-moe`) and run **evolutionary merge search** (`mergekit-evolve`).
- Reference: TIES-Merging, Yadav et al., arXiv 2306.01708.